# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

> 풀이 데이터: 이 폴더의 `data/` CSV. RavenStack은 River @ Rivalytics의 합성 교육용 데이터다. 실제 사업 성과로 일반화하지 않는다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [2]:
from pathlib import Path
import pandas as pd
DATA_DIR = Path('data')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib import font_manager
if 'Malgun Gothic' in {f.name for f in font_manager.fontManager.ttflist}:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams.update({'axes.unicode_minus': False, 'figure.figsize': (8,4), 'font.size': 11})
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
def show_rates(frame):
    display(frame.style.format({c: '{:.2%}' for c in frame.columns if 'rate' in str(c) or 'conversion' in str(c)}, na_rep='—'))
def bars(series, title, ylabel, percent=False):
    ax = (series * (100 if percent else 1)).plot.bar(color='#32689b', rot=0)
    ax.set(title=title, ylabel=ylabel, xlabel='')
    ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.show()
def make_funnel(counts, labels):
    f = pd.DataFrame({'count': counts}, index=labels)
    prev = f['count'].shift().replace(0, np.nan)
    f['conversion_rate'] = f['count'] / prev
    f['drop_rate'] = 1 - f.conversion_rate
    f['drop_count'] = f['count'].shift() - f['count']
    return f

accounts = pd.read_csv(DATA_DIR / 'ravenstack_accounts.csv')
subscriptions = pd.read_csv(DATA_DIR / 'ravenstack_subscriptions.csv')
tables = {'accounts':accounts,'subscriptions':subscriptions}
for name, df in tables.items():
    for col in df.columns:
        if col.endswith('_date') or col in ['submitted_at', 'closed_at']:
            df[col] = pd.to_datetime(df[col], errors='raise')
    print(name, df.shape)
    display(df.head())
    display(pd.DataFrame({'dtype':df.dtypes.astype(str), 'missing':df.isna().sum()}))
if 'subscriptions' in tables:
    assert subscriptions.subscription_id.is_unique
    display(subscriptions.mrr_amount.describe())
    display(subscriptions.plan_tier.value_counts())
if 'feature_usage' in tables:
    print('usage_id 중복:', feature_usage.usage_id.duplicated().sum())
    print('구독 수 / 이벤트 행 수:', subscriptions.subscription_id.nunique(), len(feature_usage))
    display(feature_usage.feature_name.value_counts())
    display(feature_usage.usage_count.describe())
    assert feature_usage.subscription_id.isin(subscriptions.subscription_id).all()

accounts (500, 10)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


,dtype,missing
account_id,str,0
account_name,str,0
industry,str,0
country,str,0
signup_date,datetime64[us],0
referral_source,str,0
plan_tier,str,0
seats,int64,0
is_trial,bool,0
churn_flag,bool,0


subscriptions (5000, 14)


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaT,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaT,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaT,Enterprise,27,5373,64476,False,False,False,False,monthly,True


,dtype,missing
subscription_id,str,0
account_id,str,0
start_date,datetime64[us],0
end_date,datetime64[us],4514
plan_tier,str,0
seats,int64,0
mrr_amount,int64,0
arr_amount,int64,0
is_trial,bool,0
upgrade_flag,bool,0


count     5000.000000
mean      2267.749400
std       3421.375348
min          0.000000
25%        285.000000
50%        931.000000
75%       2786.000000
max      33830.000000
Name: mrr_amount, dtype: float64

plan_tier
Enterprise    1723
Pro           1675
Basic         1602
Name: count, dtype: int64

---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [3]:
active_paid_mask = subscriptions.end_date.isna() & ~subscriptions.is_trial
current_paid_mrr = subscriptions.loc[active_paid_mask, 'mrr_amount'].sum()
subscription_churn_rate = subscriptions.churn_flag.mean()
cumulative_accounts = accounts.account_id.nunique()
print(f'현재 유료 구독 MRR: {current_paid_mrr:,.2f} / 구독 이탈률: {subscription_churn_rate:.2%} / 누적 가입: {cumulative_accounts:,}개')

현재 유료 구독 MRR: 10,159,608.00 / 구독 이탈률: 9.72% / 누적 가입: 500개


### 필수 1 답변
- **Q1.** 현재 유료 MRR 10,159,608.00는 유지 중인 유료 구독의 반복 매출을 직접 보여주므로 KPI다.
- **Q2.** 구독 이탈률 9.72%는 매출 유지 위험을 설명하는 보조 지표다. 전체 기록의 이탈 비율이며 월간 이탈률은 아니다.
- **Q3.** 누적 가입 500개는 이탈·무료 체험·매출을 반영하지 않아 현재 목표에서는 허무 지표 후보다. 가입 확보 자체가 목표라면 유용할 수 있다.

---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다. `Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [ ]:
subscriptions['is_active_paid'] = active_paid_mask
plan_metrics = subscriptions.groupby('plan_tier').agg(total_subscriptions=('subscription_id','nunique'), churned=('churn_flag','sum'), churn_rate=('churn_flag','mean'), active_paid=('is_active_paid','sum'))
plan_metrics['active_mrr'] = subscriptions.loc[active_paid_mask].groupby('plan_tier').mrr_amount.sum().reindex(plan_metrics.index,fill_value=0)
display(plan_metrics.style.format({'churn_rate':'{:.2%}', 'active_mrr':'{:,.2f}'}))
bars(plan_metrics.active_mrr, '요금제별 현재 활성 유료 MRR', 'MRR')

### 필수 2 답변
- **Q1.** 요금제별로 나누면 담당 상품과 고객군을 지정하여 개선할 수 있다.
- **Q2.** MRR 최대는 Enterprise (7,546,876.00)이다.
- **Q3.** 이탈률 최대는 Enterprise (9.98%)이다.
- **Q4.** 매출 기여가 가장 큰 Enterprise를 우선 점검한다. 요금제별 해지 사유와 사용 감소를 확인하여 매출 손실 위험을 줄인다. 이탈률 차이만으로 요금제의 인과 효과를 단정하지 않는다.

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [ ]:
source_metrics = accounts.groupby('referral_source').agg(accounts=('account_id','nunique'), churned=('churn_flag','sum'), churn_rate=('churn_flag','mean')).sort_values('churn_rate',ascending=False)
show_rates(source_metrics)
bars(source_metrics.churn_rate, '유입 경로별 고객사 이탈률', '이탈률 (%)', True)

### 과제 1 답변
- **Q1.** event (30.21%)이다.
- **Q2.** 해당 경로의 광고 약속과 실제 온보딩 경험의 불일치를 점검하고 유입 메시지 개선을 검증한다.
- **Q3.** 캠페인 변경으로 연결되어 행동 가능성이 있고, 동일한 고객사 단위라 이해하기 쉽다. 비교 가능성은 가입 시점·관찰 기간·고객 구성까지 맞추어야 확보된다.
- **Q4.** 고객 이탈 감소 목적에서는 기간별 고객 이탈률이 KPI, 초기 기능 활성률이 보조 지표다. 누적 가입 수는 기존 고객의 이탈을 가려 허무 지표가 될 수 있다. KPI는 목표 자체, 보조 지표는 변화의 진단, 허무 지표는 목표와 연결되지 않은 규모 수치다.

---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?
2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?
3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?
4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?
5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?

### 마무리 답변
1. 유료 구독 유지와 반복 매출 성장 문제를 분석했다.
2. 현재 활성 유료 MRR을 KPI로 선택했다.
3. 요금제별 구독 이탈률과 유입 경로별 고객 이탈률을 확인했다.
4. 누적 가입 수만 보면 이탈과 매출 감소를 놓친다.
5. 위 표의 MRR 기여와 이탈률·표본 수로 개선 대상을 정했다.